# YZM 714 — Seçenek B Demo

**Sağlık YZ Etik & Hukuki Değerlendirme Sistemi**

Bu notebook, backend sunucusu çalışmasa bile sistemi **in-process** olarak test eder. PDF SSS s.9'da belirtilen Jupyter Notebook tabanlı demo gereksinimini karşılar.

## Çalıştırma Önkoşulları

```bash
cd backend
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
python scripts/ingest_all.py   # ChromaDB doldurma (bir defalık)
```

Ardından bu notebook'u jupyter ile aç:
```bash
jupyter lab demo.ipynb
```

## 1. Ortam Kurulumu

In [ ]:
import asyncio
import json
import os
import sys
from pathlib import Path

# Backend modülünü import edebilmek için path ekle
sys.path.insert(0, str(Path.cwd()))

# .env dosyasını yükle (basit yöntem)
env_path = Path('.env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if line and not line.startswith('#') and '=' in line:
            key, val = line.split('=', 1)
            os.environ.setdefault(key.strip(), val.strip())

print('LLM Provider:', os.environ.get('LLM_PROVIDER', 'claude'))
print('Embedding model:', os.environ.get('EMBEDDING_MODEL', 'intfloat/multilingual-e5-large'))

## 2. Pipeline ve Use Case'leri Yükle

In [ ]:
from app.rag.pipeline import EvaluationPipeline
from app.api.use_cases import PRESET_USE_CASES
from app.models.schemas import EvaluationRequest, LLMProvider
from app.rules import RuleEngine

pipeline = EvaluationPipeline()
rule_engine = RuleEngine()

print(f'✓ Pipeline hazır')
print(f'✓ Kural motoru: {len(rule_engine.all_rule_definitions())} kural yüklendi')
print(f'✓ Hazır use case sayısı: {len(PRESET_USE_CASES)}')
for uc_id, uc in PRESET_USE_CASES.items():
    print(f'  • {uc_id}: {uc.title[:70]}...')

## 3. UC1 — Radyoloji CADx (Kural Motoru AÇIK)

In [ ]:
uc1 = PRESET_USE_CASES['uc1-radyoloji']

request = EvaluationRequest(
    use_case=uc1,
    llm_provider=LLMProvider.CLAUDE,  # API key yoksa LLMProvider.OLLAMA
    rules_enabled=True,
)

result_uc1_on = await pipeline.evaluate(request)

print(f'Evaluation ID: {result_uc1_on.evaluation_id}')
print(f'Risk Sınıfı:   {result_uc1_on.legal_compliance.eu_ai_act_risk_class}')
print(f'Ortalama:      {result_uc1_on.ethics_scores.average} / 10')
print(f'İhlal sayısı:  {len(result_uc1_on.rule_violations)}')
print(f'Süre:          {result_uc1_on.metadata.duration_ms} ms\n')

print('Boyut skorları:')
for dim in ['fairness', 'transparency', 'accountability', 'privacy', 'human_oversight']:
    detail = getattr(result_uc1_on.ethics_scores, dim)
    print(f'  {dim:18s} {detail.score:.1f}/10  → {detail.rationale[:80]}...')

## 4. UC1 — Radyoloji CADx (Kural Motoru KAPALI)

Aynı use case, kural motoru olmadan. Karşılaştırma için.

In [ ]:
request_off = EvaluationRequest(
    use_case=uc1,
    llm_provider=LLMProvider.CLAUDE,
    rules_enabled=False,
)

result_uc1_off = await pipeline.evaluate(request_off)

print('=== Kural motoru kapalı modu ===')
print(f'Ortalama: {result_uc1_off.ethics_scores.average} / 10')
print(f'Δ (OFF − ON): {result_uc1_off.ethics_scores.average - result_uc1_on.ethics_scores.average:+.2f}')
print(f'İhlal sayısı: {len(result_uc1_off.rule_violations)}')

## 5. UC2 — Sepsis CDSS

In [ ]:
uc2 = PRESET_USE_CASES['uc2-cdss']
request_uc2 = EvaluationRequest(use_case=uc2, llm_provider=LLMProvider.CLAUDE, rules_enabled=True)
result_uc2 = await pipeline.evaluate(request_uc2)

print(f'UC2 Ortalama: {result_uc2.ethics_scores.average} / 10')
print(f'İhlal sayısı: {len(result_uc2.rule_violations)}')
print('\nTetiklenen kurallar:')
for v in result_uc2.rule_violations:
    print(f'  [{v.severity.value.upper():7s}] {v.rule_id}  {v.rule_name}')

## 6. UC3 — Federated Learning + Sentetik Veri

In [ ]:
uc3 = PRESET_USE_CASES['uc3-veri']
request_uc3 = EvaluationRequest(use_case=uc3, llm_provider=LLMProvider.CLAUDE, rules_enabled=True)
result_uc3 = await pipeline.evaluate(request_uc3)

print(f'UC3 Ortalama: {result_uc3.ethics_scores.average} / 10')
print(f'İhlal sayısı: {len(result_uc3.rule_violations)}')
print(f'\nRisk sınıfı: {result_uc3.legal_compliance.eu_ai_act_risk_class}')
print(f'Uygulanabilir düzenlemeler: {len(result_uc3.legal_compliance.applicable_regulations)}')
for reg in result_uc3.legal_compliance.applicable_regulations:
    print(f'  - {reg}')

## 7. Ablation Özet Tablosu

In [ ]:
import pandas as pd

# Her use case için kural-açık/kapalı koş
results = []
for uc_id, uc in PRESET_USE_CASES.items():
    for rules_on in (True, False):
        req = EvaluationRequest(use_case=uc, llm_provider=LLMProvider.CLAUDE, rules_enabled=rules_on)
        r = await pipeline.evaluate(req)
        results.append({
            'use_case': uc_id,
            'mode': 'ON' if rules_on else 'OFF',
            'avg': r.ethics_scores.average,
            'fairness': r.ethics_scores.fairness.score,
            'transparency': r.ethics_scores.transparency.score,
            'privacy': r.ethics_scores.privacy.score,
            'risk_class': r.legal_compliance.eu_ai_act_risk_class.value,
            'violations': len(r.rule_violations),
            'duration_ms': r.metadata.duration_ms,
        })

df = pd.DataFrame(results)
df

## 8. JSON Çıktı Örneği

In [ ]:
# UC1 sonucu tam JSON olarak
print(json.dumps(
    json.loads(result_uc1_on.model_dump_json()),
    indent=2,
    ensure_ascii=False,
)[:2000])

## 9. Kendi Use Case'in

Aşağıdaki hücreyi düzenleyerek kendi sağlık YZ uygulamanı değerlendirebilirsin.

In [ ]:
from app.models.schemas import UseCase, HealthcareArea

custom_uc = UseCase(
    title='Otonom triaj chatbot — kötü tasarım örneği',
    area=HealthcareArea.CDSS,
    description=(
        'GPT-4o tabanlı chatbot hastaların semptomlarına göre acil serviste '
        'triaj önceliğini otonom olarak belirliyor. Hekim onayı gerektirmiyor. '
        'EHR ve hasta ses verisi kullanılıyor. KVKK için hash\'leme yapılıyor.'
    ),
    affected_stakeholders=['hasta', 'hemşire'],
    jurisdiction=['TR', 'EU', 'IT'],
)

custom_request = EvaluationRequest(
    use_case=custom_uc,
    llm_provider=LLMProvider.CLAUDE,
    rules_enabled=True,
)

custom_result = await pipeline.evaluate(custom_request)

print(f'Bu tasarım için ortalama skor: {custom_result.ethics_scores.average} / 10')
print(f'\nTetiklenen ihlaller ({len(custom_result.rule_violations)}):')
for v in custom_result.rule_violations:
    icon = '🔴' if v.severity.value == 'error' else ('🟡' if v.severity.value == 'warning' else 'ℹ')
    print(f'  {icon} [{v.rule_id}] {v.rule_name}')
    print(f'     → {v.message[:120]}...')

---

## Sonuç

Bu notebook, sistemin **3 hazır use case + 1 özel use case** üzerinde nasıl çalıştığını gösterir. Kural motoru açık/kapalı karşılaştırması, kuralların **konservatif kalibrasyon** sağladığını ve **somut düzenleyici geribildirim** ürettiğini doğrular.

Sistemin tam web arayüzü için: `docker compose up` → http://localhost:3000